# Session 21: Text as data and meaning as numbers

Nine tasks on the 800 Lakeside Mart feedback messages. Cleaning, the regular expression, keyword scoring and cosine run anywhere. The classifier and the meaning search need scikit-learn and sentence-transformers, which Colab has.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Load the messages
Read `feedback.csv` into `fb` and print its shape.

In [ ]:
import pandas as pd

fb = pd.read_csv('feedback.csv')
print(fb.shape)

In [ ]:
assert fb.shape == (800, 6), 'fb should have 800 rows and 6 columns. Check the file name'
print('Task 1 correct')

## Task 2. Clean the text and count words
Make a column `clean` (lower case, letters and spaces only) and a column `n_words`. Store the mean word count in `avg_words`.

In [ ]:
fb['clean'] = (fb.message.str.lower()
               .str.replace(r'[^a-z\s]', '', regex=True))
fb['n_words'] = fb.clean.str.split().str.len()
avg_words = fb.n_words.mean()
print(round(avg_words, 1))

In [ ]:
assert abs(avg_words - 11.9) < 0.1, 'avg_words should be about 11.9. Check n_words counts the words of the clean text, and avg_words is its mean'
print('Task 2 correct')

## Task 3. Strip phone numbers
Use `re.sub` with a pattern to replace each phone number in `raw` with `[PHONE]`. Store the cleaned messages in `safe`. (The numbers are made up.)

In [ ]:
import re

raw = ['Call 0700 000 111 about my refund.',
       'Sent from +256 700 000 222 twice.',
       'No number in this one.']
pat = r'(\+?256|0)[\s-]?\d{3}[\s-]?\d{3}[\s-]?\d{3}'
safe = [re.sub(pat, '[PHONE]', r) for r in raw]
print(safe)

In [ ]:
assert safe[0] == 'Call [PHONE] about my refund.', 'The pattern must match 0700 000 111'
assert safe[1] == 'Sent from [PHONE] twice.', 'The pattern must match +256 700 000 222'
assert safe[2] == raw[2], 'A message with no number stays the same'
print('Task 3 correct')

## Task 4. Score a keyword search
The `theme` column is the answer key. Store the number of messages with theme `queue` in `n_theme`, and how many of them contain the word `queue` in `n_found`. Store the precision at ten of the keyword hits in `p10`: the share of the first ten messages containing `queue` whose theme is `queue`. Store the recall in `recall` (`n_found` divided by `n_theme`). Finally widen the keyword to `wait` (use `case=False`) and store in `p_wait` the share of those hits whose theme is `queue`.

In [ ]:
q = fb[fb.theme == 'queue']
n_theme = len(q)
n_found = q.message.str.contains('queue').sum()
hits = fb[fb.message.str.contains('queue')]
p10 = (hits.head(10).theme == 'queue').mean()
recall = n_found / n_theme
wait = fb[fb.message.str.contains('wait', case=False)]
p_wait = (wait.theme == 'queue').mean()
print(n_theme, n_found, p10, round(recall, 2), round(p_wait, 2))

In [ ]:
assert n_theme == 89, 'Filter fb on theme == queue'
assert n_found == 19, 'Count the queue-theme messages that contain the word queue'
assert p10 == 1.0, 'Take the first ten hits with head(10)'
assert abs(recall - 0.213) < 0.01, 'recall is n_found divided by n_theme'
assert abs(p_wait - 0.667) < 0.01, 'Use str.contains with case=False'
print('Task 4 correct')

## Task 5. A sentiment classifier
Split `fb.message` and `fb.sentiment` 75 / 25 with `random_state=42`. Fit `TfidfVectorizer` and `LogisticRegression`, store the test predictions in `pred`, store the test accuracy in `acc`, and keep the test rows in `test` with `fb.loc[X_te.index]`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    fb.message, fb.sentiment, test_size=0.25, random_state=42)
vec = TfidfVectorizer()
model = LogisticRegression(max_iter=1000)
model.fit(vec.fit_transform(X_tr), y_tr)
pred = model.predict(vec.transform(X_te))
acc = (pred == y_te).mean()
test = fb.loc[X_te.index]
print(round(acc, 3))

In [ ]:
assert 0.85 <= acc <= 1.0, 'acc should be between 0.85 and 1.0. Fit the vectoriser and model on the training messages only, then score the test messages'
print('Task 5 correct')

## Task 6. Read the mistakes
Keep the rows of `test` where the true label differs from `pred` in `wrong`. Read the first ten. Which ones look like noisy labels?

In [ ]:
# pred comes from the sklearn model above
wrong = test[test.sentiment != pred]
print(len(wrong))
wrong[['message', 'sentiment']].head(10)

In [ ]:
assert len(wrong) == (pred != y_te).sum(), 'wrong should hold only the rows where label and pred differ'
assert len(wrong) < 0.15 * len(test), 'A good model gets most messages right'
print('Task 6 correct')

## Task 7. Cosine similarity by hand
Write `cosine(x, y)` with numpy. Check it on two vectors. Store `cosine(a, b)` in `sim`.

In [ ]:
import numpy as np

def cosine(x, y):
    return x @ y / (np.linalg.norm(x) * np.linalg.norm(y))

a = np.array([3, 1, 0])
b = np.array([2, 1, 1])
sim = cosine(a, b)
print(round(sim, 2))

In [ ]:
assert abs(sim - 0.9) < 0.01, 'sim should be about 0.90: the dot product of a and b, divided by the length of a times the length of b'
print('Task 7 correct')

## Task 8. Search by meaning
Encode the messages with `all-MiniLM-L6-v2` (it downloads about 90 MB once). Search for `long queue at the till`, but only among the messages that do not contain the word `queue`. Keep the best ten in `top10` and count how many have theme `queue` in `n_right`. Then try your own question.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
vecs = model.encode(fb.message.tolist(), normalize_embeddings=True)
q = model.encode(['long queue at the till'], normalize_embeddings=True)
fb['score'] = (vecs @ q.T).ravel()
missed = fb[~fb.message.str.contains('queue')]
top10 = missed.sort_values('score', ascending=False).head(10)
n_right = (top10.theme == 'queue').sum()
print(n_right)
top10[['message', 'score']]

In [ ]:
assert n_right >= 6, 'n_right should be at least 6. Search only the messages without the word queue, rank them by score, and count the theme queue rows in the best ten'
print('Task 8 correct')

## Task 9. Where does it fail?
Search for `waiting forever to pay` by keyword (`str.contains('waiting')`) and by meaning. Write one sentence below: which found more of the 89 queue messages, and name one message type where each method would fail. Remember that local languages need testing with local speakers.